In this notebook we use the bounding boxes given with the [Imagenet data set](https://www.kaggle.com/c/imagenet-object-localization-challenge) to generate a new sub-set. From the original data set `Imagenet_full` by only keeping the smallest square comprising the bounding boxe we generate `Imagenet_bbox` sub-set




In [1]:
from retinotopy import *
welcome()

-----------------------------------------------------------------------------------------
On date 2025-05-08, Running learning on host obiwan.local with device mps, pytorch==2.7.0
-----------------------------------------------------------------------------------------
Welcome on macOS-15.4.1-arm64-arm-64bit-Mach-O


In [2]:
args = Params()
source_data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{source_data_set_type}' # Directory containing original images
target_data_set_type = 'bbox'
args.target_root = f'{DATAROOT}/Imagenet_{target_data_set_type}' # Directory containing cropped images
os.makedirs(args.target_root, exist_ok=True)
args.folders, args.root, args.target_root

(['val', 'train'],
 '/Volumes/SSD1TO/Deep_learning/data/Imagenet_full',
 '/Volumes/SSD1TO/Deep_learning/data/Imagenet_bbox')

## reading localisation metadata

First for the 'train' dataset:

In [3]:
def get_boxes(df, value):
    idx = list(df['ImageId'][df['ImageId'] == value].index)
    bboxes = []
    if idx:
        for i in range(len(df["PredictionString"][idx[0]].split(' '))//5):
            pos =(5*i)
            bboxes.append({'xmin' : int(df["PredictionString"][idx[0]].split(' ')[1 + pos]),
                           'ymin' : int(df["PredictionString"][idx[0]].split(' ')[2 + pos]),
                           'xmax' : int(df["PredictionString"][idx[0]].split(' ')[3 + (5 *i)]),
                           'ymax' : int(df["PredictionString"][idx[0]].split(' ')[4 + (5 *i)])
                                        })
    return bboxes

In [4]:
with open(args.annotations_train, 'r') as csv_file:
    df_data = pd.read_csv(csv_file)
df_data.head()

,ImageId,PredictionString
0,n02017213_7894,n02017213 115 49 448 294
1,n02017213_7261,n02017213 91 42 330 432
2,n02017213_5636,n02017213 230 104 414 224
3,n02017213_6132,n02017213 46 82 464 387
4,n02017213_7659,n02017213 103 66 331 335


In [5]:
get_boxes(df_data, 'n02099849_2300')

[{'xmin': 151, 'ymin': 146, 'xmax': 332, 'ymax': 333},
 {'xmin': 7, 'ymin': 232, 'xmax': 331, 'ymax': 467}]

In [6]:
get_boxes(df_data, 'n01440764_32420')

[]

Now for the 'val' dataset:

In [7]:
with open(args.annotations_val, 'r') as csv_file:
    df_data = pd.read_csv(csv_file)
df_data.head()

,ImageId,PredictionString,origin_size
0,ILSVRC2012_val_00048981,n03995372 85 1 499 272,"(500, 360)"
1,ILSVRC2012_val_00037956,n03481172 131 0 499 254,"(500, 333)"
2,ILSVRC2012_val_00026161,n02108000 38 0 464 280,"(500, 334)"
3,ILSVRC2012_val_00026171,n03109150 0 14 216 299,"(225, 300)"
4,ILSVRC2012_val_00008726,n02119789 255 142 454 329 n02119789 44 21 322 ...,"(500, 357)"


In [8]:
get_boxes(df_data, 'ILSVRC2012_val_00026171')

[{'xmin': 0, 'ymin': 14, 'xmax': 216, 'ymax': 299}]

In [9]:
def clean_list(list_dir, patterns=['.DS_Store', '.ipynb_checkpoints']):
    for pattern in patterns:
        if pattern in list_dir: list_dir.remove(pattern)
    return list_dir

## cropping images

In [12]:
from PIL import Image 

def square_box(xmin, ymin, xmax, ymax):
    temp = ((xmax-xmin)-(ymax-ymin))//2 # signed radius
    if temp > 0 :
        ymin -= temp
        ymax += temp
    else:
        xmin += temp
        xmax -= temp
    return xmin, ymin, xmax, ymax


for folder in args.folders :
    # first level
    print(f'\nFolder \"{folder}\"')
    source_folder = os.path.join(args.root, folder)
    boxes_folder = os.path.join(args.target_root, folder)
    os.makedirs(boxes_folder, exist_ok=True)

    # second level
    with open(f'data/LOC_{folder}_solution.csv', 'r') as csv_file:
        df_data = pd.read_csv(csv_file)
    for i_label, label_id in enumerate(args.loader):
        print(f'Scraping images for id \"{label_id}\" : {labels[i_label]} ', end='')
        img_source_folder = os.path.join(source_folder, label_id)
        target_folder = os.path.join(boxes_folder, label_id)
        if True: #not os.path.isdir(target_folder):
            os.makedirs(target_folder, exist_ok=True)
            print(os.listdir(img_source_folder))
            for imgs in  clean_list(os.listdir(img_source_folder)):
                data_local = os.path.join(img_source_folder, imgs)
                objects = get_boxes(df_data, imgs.split('.')[0])

                original_image = Image.open(data_local, mode='r').convert('RGB')
                for i_obj, object in enumerate(objects): #if len(obj)  > 0 :
                    xmin = object['xmin']
                    ymin = object['ymin']
                    xmax = object['xmax']
                    ymax = object['ymax']

                    crop_image = original_image.crop(square_box(xmin, ymin, xmax, ymax))
                    no = '' if i_obj==0 else f'_{i_obj}'
                    img_name = imgs.split('.')[0] + no + '.jpg'
                    crop_image.save(os.path.join(target_folder, img_name))

        print(f' - in:  {len(clean_list(os.listdir(img_source_folder)))} / out:  {len(clean_list(os.listdir(target_folder)))}')


Folder "val"
Scraping images for id "d" : tench 

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/SSD1TO/Deep_learning/data/Imagenet_full/val/d'

In [14]:
args.loader


'data/Imagenet_urls_ILSVRC_2016.json'